# Нэгдүгээр эрэмбийн логик: гэр бүлийн мэдлэгийн сан

**pyDatalog ба Prolog | Үндсэн болон нэмэлт даалгаврын гүйцэтгэл**

Эх сурвалж: `Lab6_FOL_Заавар.pdf`. PDF-ийн дотор лаборатори 4 гэж дугаарлаж,
тайланг `Lab4_FOL.ipynb`, Prolog кодыг `family.pl` гэж нэрлэхийг шаардсан тул
тэр нэршлийг баримталсан.

Энд 15 хүн, 18 эцэг эхийн холбооноос дүрмээр шинэ мэдлэг гаргана.
`parent(P, C)`-ийн эхний аргумент нь эцэг эх, хоёр дахь нь хүүхэд.
`uncle` ба `aunt` нь насны ялгаа заахгүй, эцэг болон эхийн талын цусан төрлийн
ах, эгч, дүүг хамруулна. Хуримын ургийг нэмэлт дүрмээр тусад нь тодорхойлно.

**Ажиллуулах:** төслийн `.venv` орчны Python kernel сонгоод **Restart Kernel and Run All Cells**.
Эхний cell шаардлагатай санг зөвхөн суулгаагүй үед суулгана.
Алхам бүрийн `assert` шалгалт амжилттай бол `OK` хэвлэнэ. Нэмэлт гишүүнийг
туршилтын дараа хасаж, үндсэн 15 хүний санг сэргээнэ.

## Алхам 1. Орчин, баримт, дүрэм, асуулт

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys

if importlib.util.find_spec('pyDatalog') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyDatalog==0.22.4'])

from pyDatalog import pyDatalog
pyDatalog.clear()

def answers(query):
    # Хариуг давхардалгүй, эрэмбэлсэн tuple жагсаалтаар авна.
    result = pyDatalog.ask(query)
    return [] if result is None else sorted(result.answers)

def count(query):
    # Хариугүй үед ask нь None тул 0 буцаана.
    result = pyDatalog.ask(query)
    return 0 if result is None else len(result.answers)

def names(query):
    return [row[0] for row in answers(query)]

checks = []
def check(label, actual, expected):
    assert actual == expected, f'{label}: {actual!r} != {expected!r}'
    checks.append(label)
    print(f'OK | {label}: {actual}')

pyDatalog.create_terms('bird, flies, X')
for attempt in (1, 2):
    pyDatalog.clear()
    # Баримт: tweety, sparrow хоёр шувуу.
    +bird('tweety')
    +bird('sparrow')
    # Шувуу ниснэ: ∀x (bird(x) → flies(x)).
    flies(X) <= bird(X)
    check(f'Жижиг жишээ, давталт {attempt}', names('flies(X)'), ['sparrow', 'tweety'])

pyDatalog.clear()  # Жишээг арилгаад гэр бүлийн санг эхлүүлнэ.
pyDatalog.create_terms('''male, female, parent, person, has_child, childless,
father, mother, son, daughter, grandparent, sibling, sibling_unchecked,
brother, sister, uncle, aunt, cousin, ancestor, descendant,
great_grandparent, has_cousin, has_grandchild, spouse, uncle_in_law,
aunt_in_law, X, Y, Z, P, Q''')
print('Python:', sys.version.split()[0])
print('pyDatalog:', importlib.metadata.version('pyDatalog'))

OK | Жижиг жишээ, давталт 1: ['sparrow', 'tweety']
OK | Жижиг жишээ, давталт 2: ['sparrow', 'tweety']
Python: 3.12.14
pyDatalog: 0.22.4


**Баримт** нь шууд үнэн гэж оруулсан мэдээлэл: `parent('bataa', 'bold')`.
**Дүрэм** нь нөхцөлөөс дүгнэлт гаргана: `father(X,Y) <= parent(X,Y) & male(X)`.
**Асуулт** нь тохирох утгыг олно: `father(X,'anar')`.
Дүрэм нь нэг эерэг толгойтой Horn хэлбэртэй. Үгүйсгэлтэй `childless` нь
цэвэр эерэг Horn дүрмийн өргөтгөл бөгөөд closed-world assumption хэрэглэнэ.

## Алхам 2. Гэр бүлийн баримт

| Эцэг эх | Хүүхдүүд |
|---|---|
| bataa, saraa | bold, oyuna, tomor |
| bold, tsetseg | anar, zolboo |
| dorj, oyuna | munkh, naraa |
| tomor, ganaa | enkhee |
| bat, anar | temuulen |

Энэ хүснэгт нь холбооны бүтцийг бүрэн харуулна. `bat` нь `temuulen`-ийн эцэг,
`anar` нь эх; `bat`-ийн эцэг эх өгөгдөөгүй. Түүний хүснэгтэд бичигдсэн үеийг
эцэг эхийн холбоо гэж ойлгож болохгүй.

In [2]:
MALES = ('bataa', 'bold', 'tomor', 'dorj', 'zolboo', 'munkh', 'temuulen', 'bat')
FEMALES = ('saraa', 'oyuna', 'tsetseg', 'ganaa', 'anar', 'naraa', 'enkhee')
FAMILIES = (
    (('bataa', 'saraa'), ('bold', 'oyuna', 'tomor')),
    (('bold', 'tsetseg'), ('anar', 'zolboo')),
    (('dorj', 'oyuna'), ('munkh', 'naraa')),
    (('tomor', 'ganaa'), ('enkhee',)),
    (('bat', 'anar'), ('temuulen',)),
)
for name in MALES:
    +male(name)
for name in FEMALES:
    +female(name)
for parents, children in FAMILIES:
    for p in parents:
        for c in children:
            +parent(p, c)

check('Эрэгтэй', count('male(X)'), 8)
check('Эмэгтэй', count('female(X)'), 7)
check('Эцэг эхийн холбоо', count('parent(X,Y)'), 18)
check('Bataa-ийн хүүхдүүд', names("parent('bataa',X)"), ['bold', 'oyuna', 'tomor'])
check('Хариугүй асуултын тоо', count("parent('bataa','bat')"), 0)

OK | Эрэгтэй: 8
OK | Эмэгтэй: 7
OK | Эцэг эхийн холбоо: 18
OK | Bataa-ийн хүүхдүүд: ['bold', 'oyuna', 'tomor']
OK | Хариугүй асуултын тоо: 0


## Алхам 3. Энгийн дүрмүүд

In [3]:
# X нь Y-ийн эцэг: ∀x∀y ((parent(x,y) ∧ male(x)) → father(x,y)).
father(X,Y) <= parent(X,Y) & male(X)
# X нь Y-ийн эх: ∀x∀y ((parent(x,y) ∧ female(x)) → mother(x,y)).
mother(X,Y) <= parent(X,Y) & female(X)
# X нь Y-ийн хүү: ∀x∀y ((parent(y,x) ∧ male(x)) → son(x,y)).
son(X,Y) <= parent(Y,X) & male(X)
# X нь Y-ийн охин: ∀x∀y ((parent(y,x) ∧ female(x)) → daughter(x,y)).
daughter(X,Y) <= parent(Y,X) & female(X)
# Эрэгтэй бол хүн: ∀x (male(x) → person(x)).
person(X) <= male(X)
# Эмэгтэй бол хүн: ∀x (female(x) → person(x)). Хоёр дүрэм нь OR утгатай.
person(X) <= female(X)

check('Anar-ийн эцэг', names("father(X,'anar')"), ['bold'])
check('Bold-ийн эх', names("mother(X,'bold')"), ['saraa'])
check('Oyuna-ийн хүү', names("son(X,'oyuna')"), ['munkh'])
check('Oyuna-ийн охин', names("daughter(X,'oyuna')"), ['naraa'])
check('Эцэг холбооны тоо', count('father(X,Y)'), 9)
check('Эх холбооны тоо', count('mother(X,Y)'), 9)
check('Хүний тоо', count('person(X)'), 15)

OK | Anar-ийн эцэг: ['bold']
OK | Bold-ийн эх: ['saraa']
OK | Oyuna-ийн хүү: ['munkh']
OK | Oyuna-ийн охин: ['naraa']
OK | Эцэг холбооны тоо: 9
OK | Эх холбооны тоо: 9
OK | Хүний тоо: 15


## Алхам 4. Өвөө эмээ, ах дүү

In [4]:
# X нь Y-ийн өвөө/эмээ: ∀x∀y∀z ((parent(x,z) ∧ parent(z,y)) → grandparent(x,y)).
grandparent(X,Y) <= parent(X,Z) & parent(Z,Y)
# Нэг эцэг эхтэй, өөр хүмүүс: ∀x∀y∀p ((parent(p,x) ∧ parent(p,y) ∧ x≠y) → sibling(x,y)).
sibling(X,Y) <= parent(P,X) & parent(P,Y) & (X != Y)

check('Temuulen-ийн өвөө эмээ', names("grandparent(X,'temuulen')"), ['bold', 'tsetseg'])
check('Өвөө эмээгийн холбоо', count('grandparent(X,Y)'), 12)
check('Munkh-ийн ах дүү', names("sibling(X,'munkh')"), ['naraa'])
check('Bold-ийн ах дүү', names("sibling(X,'bold')"), ['oyuna', 'tomor'])
check('Ах дүүгийн эрэмбэлсэн хос', count('sibling(X,Y)'), 10)
check('Өөрийн ах дүү болохгүй', count('sibling(X,X)'), 0)

# Зөв дүрмийг эвдэхгүйгээр X≠Y нөхцөлийг хассан хувилбарыг туршина.
# ∀x∀y∀p ((parent(p,x) ∧ parent(p,y)) → sibling_unchecked(x,y)).
sibling_unchecked(X,Y) <= parent(P,X) & parent(P,Y)
check('X≠Y байхгүй үеийн хос', count('sibling_unchecked(X,Y)'), 19)
check('Нэмэгдсэн өөртэйгөө холбоо', names('sibling_unchecked(X,X)'),
      ['anar', 'bold', 'enkhee', 'munkh', 'naraa', 'oyuna', 'temuulen', 'tomor', 'zolboo'])

OK | Temuulen-ийн өвөө эмээ: ['bold', 'tsetseg']
OK | Өвөө эмээгийн холбоо: 12
OK | Munkh-ийн ах дүү: ['naraa']
OK | Bold-ийн ах дүү: ['oyuna', 'tomor']
OK | Ах дүүгийн эрэмбэлсэн хос: 10
OK | Өөрийн ах дүү болохгүй: 0
OK | X≠Y байхгүй үеийн хос: 19
OK | Нэмэгдсэн өөртэйгөө холбоо: ['anar', 'bold', 'enkhee', 'munkh', 'naraa', 'oyuna', 'temuulen', 'tomor', 'zolboo']


`grandparent`-ийн Horn өгүүлбэрт бүх хувьсагч, түүний дотор **z нь ∀**-аар
холбогдоно. Өөрөөр хэлбэл дурын тохирох `z` олдвол дүгнэлт үнэн.
Үүнийг **∀x∀y ((∃z (parent(x,z) ∧ parent(z,y))) → grandparent(x,y))**
гэж эквивалент бичиж болно. Асуултад `z` нь завсрын хүнийг олох оршин байх
гэрчийн үүрэгтэй; `z` толгойд гарахгүй. Энэ нь `∀z` бүх хүн заавал завсрын
эцэг эх байна гэсэн шаардлага биш.

`sibling` нь нэг эцэг эхээ хуваалцвал хангалттай тул хагас ах дүүг ч хамруулна.
Хоёр эцэг эхээрээ тус тус батлагдсан ч pyDatalog нэг `(x,y)` хосыг нэг удаа хадгална.
19 − 10 = **9** нэмэлт нь эцэг эх нь бүртгэгдсэн 9 хүний `(x,x)` хос юм.

## Алхам 5. Ураг төрлийн дүрмүүд

In [5]:
# Эрэгтэй ах дүү: ∀x∀y ((sibling(x,y) ∧ male(x)) → brother(x,y)).
brother(X,Y) <= sibling(X,Y) & male(X)
# Эмэгтэй ах дүү: ∀x∀y ((sibling(x,y) ∧ female(x)) → sister(x,y)).
sister(X,Y) <= sibling(X,Y) & female(X)
# Эцэг/эхийн эрэгтэй ах дүү: ∀x∀y∀p ((parent(p,y) ∧ brother(x,p)) → uncle(x,y)).
uncle(X,Y) <= parent(P,Y) & brother(X,P)
# Эцэг/эхийн эмэгтэй ах дүү: ∀x∀y∀p ((parent(p,y) ∧ sister(x,p)) → aunt(x,y)).
aunt(X,Y) <= parent(P,Y) & sister(X,P)
# Ах дүү хүмүүсийн хүүхдүүд: ∀x∀y∀p∀q ((parent(p,x) ∧ parent(q,y) ∧ sibling(p,q)) → cousin(x,y)).
cousin(X,Y) <= parent(P,X) & parent(Q,Y) & sibling(P,Q)

check('Oyuna-ийн эрэгтэй ах дүү', names("brother(X,'oyuna')"), ['bold', 'tomor'])
check('Zolboo-ийн эмэгтэй ах дүү', names("sister(X,'zolboo')"), ['anar'])
check('Anar-ийн авга/нагац ах', names("uncle(X,'anar')"), ['tomor'])
check('Anar-ийн авга/нагац эгч', names("aunt(X,'anar')"), ['oyuna'])
check('Temuulen-ийн авга/нагац ах', names("uncle(X,'temuulen')"), ['zolboo'])
check('Enkhee-ийн үеэлүүд', names("cousin(X,'enkhee')"), ['anar', 'munkh', 'naraa', 'zolboo'])
check('Anar-ийн үеэлүүд', names("cousin(X,'anar')"), ['enkhee', 'munkh', 'naraa'])
check('Үеэлийн эрэмбэлсэн хос', count('cousin(X,Y)'), 16)
check('Dorj цусан төрлийн авга биш', count("uncle('dorj','anar')"), 0)

OK | Oyuna-ийн эрэгтэй ах дүү: ['bold', 'tomor']
OK | Zolboo-ийн эмэгтэй ах дүү: ['anar']
OK | Anar-ийн авга/нагац ах: ['tomor']
OK | Anar-ийн авга/нагац эгч: ['oyuna']
OK | Temuulen-ийн авга/нагац ах: ['zolboo']
OK | Enkhee-ийн үеэлүүд: ['anar', 'munkh', 'naraa', 'zolboo']
OK | Anar-ийн үеэлүүд: ['enkhee', 'munkh', 'naraa']
OK | Үеэлийн эрэмбэлсэн хос: 16
OK | Dorj цусан төрлийн авга биш: 0


## Алхам 6. Рекурсив дүрэм

In [6]:
# Шууд эцэг эх бол өвөг: ∀x∀y (parent(x,y) → ancestor(x,y)).
ancestor(X,Y) <= parent(X,Y)
# Завсрын хүний өвгөөр дамжуулна: ∀x∀y∀z ((parent(x,z) ∧ ancestor(z,y)) → ancestor(x,y)).
ancestor(X,Y) <= parent(X,Z) & ancestor(Z,Y)
# Үр удам нь өвгийн урвуу: ∀x∀y (ancestor(y,x) → descendant(x,y)).
descendant(X,Y) <= ancestor(Y,X)

check('Temuulen-ийн өвөг дээдэс', names("ancestor(X,'temuulen')"),
      ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg'])
check('Bataa-ийн үр удам', names("descendant(X,'bataa')"),
      ['anar', 'bold', 'enkhee', 'munkh', 'naraa', 'oyuna', 'temuulen', 'tomor', 'zolboo'])
check('Bataa-ийн үр удмын тоо', count("descendant(X,'bataa')"), 9)
check('Anar-ийн өвөг дээдэс', names("ancestor(X,'anar')"), ['bataa', 'bold', 'saraa', 'tsetseg'])
check('Өвгийн холбоо', count('ancestor(X,Y)'), 32)
check('Bataa нь Temuulen-ийн өвөг', count("ancestor('bataa','temuulen')"), 1)

OK | Temuulen-ийн өвөг дээдэс: ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
OK | Bataa-ийн үр удам: ['anar', 'bold', 'enkhee', 'munkh', 'naraa', 'oyuna', 'temuulen', 'tomor', 'zolboo']
OK | Bataa-ийн үр удмын тоо: 9
OK | Anar-ийн өвөг дээдэс: ['bataa', 'bold', 'saraa', 'tsetseg']
OK | Өвгийн холбоо: 32
OK | Bataa нь Temuulen-ийн өвөг: 1


**Гараар батлах хэлхээ:** `bataa → bold → anar → temuulen`.

1. `parent(anar,temuulen)` тул суурь дүрмээр `ancestor(anar,temuulen)`.
2. `parent(bold,anar)` ба дээрх дүгнэлтээс `ancestor(bold,temuulen)`.
3. `parent(bataa,bold)` ба дээрх дүгнэлтээс `ancestor(bataa,temuulen)`.

**Хоёр дүрмийг хамтатгасан Horn томъёо:**

\[
[\forall x\forall y(parent(x,y)\to ancestor(x,y))]\land
[\forall x\forall y\forall z((parent(x,z)\land ancestor(z,y))\to ancestor(x,y))].
\]

Рекурсив хэсэгт биеийн `z`-г ∃-аар бичсэн эквивалент хэлбэр нь
`∀x∀y ((∃z(parent(x,z) ∧ ancestor(z,y))) → ancestor(x,y))`.
Эдгээр импликаци дангаараа дурын FOL загварт нэмэлт `ancestor` холбоог хориглохгүй.
Программын үр дүнг дүрмүүдийн **хамгийн бага тогтмол цэг / хамгийн бага загвар**
болгон тайлбарлахад `ancestor` нь `parent`-ийн эерэг урттай дамжих хаалт болно.

Prolog-д суурь дүрмийг эхэнд, рекурсив дүрэмд `parent`-ийг дуудлагын өмнө тавина.
Энэ сангийн эцэг эхийн граф циклгүй тул хайлт төгсөнө. Циклтэй сангийн хувьд
энэ дараалал дангаараа хангалтгүй; cycle шалгалт эсвэл tabling хэрэгтэй.

## Алхам 7. Асуулт ба аюулгүй үгүйсгэл

In [7]:
# Ядаж нэг хүүхэдтэй: ∀x∀y (parent(x,y) → has_child(x)).
has_child(X) <= parent(X,Y)
# Бүртгэлтэй хүн бөгөөд хүүхэдтэй нь батлагдахгүй бол хүүхэдгүй (CWA).
# ∀x ((person(x) ∧ ¬has_child(x)) → childless(x)).
childless(X) <= person(X) & ~has_child(X)
# Элэнц өвөө/эмээ: ∀x∀y∀z ((parent(x,z) ∧ grandparent(z,y)) → great_grandparent(x,y)).
great_grandparent(X,Y) <= parent(X,Z) & grandparent(Z,Y)
# Ядаж нэг үеэлтэй: ∀x∀y (cousin(x,y) → has_cousin(x)).
has_cousin(X) <= cousin(X,Y)
# Ядаж нэг ач/зээтэй: ∀x∀y (grandparent(x,y) → has_grandchild(x)).
has_grandchild(X) <= grandparent(X,Y)

check('Хүүхэдгүй хүмүүс', names('childless(X)'), ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo'])
check('Хүүхэдгүй хүний тоо', count('childless(X)'), 5)
check('Bataa хүүхэдгүй биш', count("childless('bataa')"), 0)
check('Temuulen хүүхэдгүй', count("childless('temuulen')"), 1)
check('Temuulen-ийн элэнц өвөө эмээ', names("great_grandparent(X,'temuulen')"), ['bataa', 'saraa'])
check('Үеэлтэй хүмүүс', names('has_cousin(X)'), ['anar', 'enkhee', 'munkh', 'naraa', 'zolboo'])
check('Ач зээтэй хүмүүс', names('has_grandchild(X)'), ['bataa', 'bold', 'saraa', 'tsetseg'])

from IPython.display import Markdown, display
query_rows = [
    ('Болдын хүүхдүүд', "parent('bold',X)", ['anar', 'zolboo']),
    ('Temuulen-ийн өвөө эмээ', "grandparent(X,'temuulen')", ['bold', 'tsetseg']),
    ('Munkh-ийн авга/нагац ах нар', "uncle(X,'munkh')", ['bold', 'tomor']),
    ('Enkhee-ийн үеэлүүд', "cousin(X,'enkhee')", ['anar', 'munkh', 'naraa', 'zolboo']),
    ('Үеэлтэй хүмүүс', 'has_cousin(X)', ['anar', 'enkhee', 'munkh', 'naraa', 'zolboo']),
    ('Хүүхэдгүй хүмүүс', 'childless(X)', ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']),
    ('Temuulen-ийн элэнц өвөө эмээ', "great_grandparent(X,'temuulen')", ['bataa', 'saraa']),
    ('Ач, зээтэй хүмүүс', 'has_grandchild(X)', ['bataa', 'bold', 'saraa', 'tsetseg']),
]
table = ['| № | Асуулт | Query | Программын хариу | Гараар шалгасан |',
         '|---|---|---|---|---|']
for index, (question, query, manual) in enumerate(query_rows, 1):
    actual = names(query)
    check(f'Хүснэгтийн асуулт {index}', actual, manual)
    table.append(f'| {index} | {question} | `{query}` | {", ".join(actual)} ({len(actual)}) | Тохирсон |')
display(Markdown('\n'.join(table)))

OK | Хүүхэдгүй хүмүүс: ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']
OK | Хүүхэдгүй хүний тоо: 5
OK | Bataa хүүхэдгүй биш: 0
OK | Temuulen хүүхэдгүй: 1
OK | Temuulen-ийн элэнц өвөө эмээ: ['bataa', 'saraa']
OK | Үеэлтэй хүмүүс: ['anar', 'enkhee', 'munkh', 'naraa', 'zolboo']
OK | Ач зээтэй хүмүүс: ['bataa', 'bold', 'saraa', 'tsetseg']
OK | Хүснэгтийн асуулт 1: ['anar', 'zolboo']
OK | Хүснэгтийн асуулт 2: ['bold', 'tsetseg']
OK | Хүснэгтийн асуулт 3: ['bold', 'tomor']
OK | Хүснэгтийн асуулт 4: ['anar', 'munkh', 'naraa', 'zolboo']
OK | Хүснэгтийн асуулт 5: ['anar', 'enkhee', 'munkh', 'naraa', 'zolboo']
OK | Хүснэгтийн асуулт 6: ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']
OK | Хүснэгтийн асуулт 7: ['bataa', 'saraa']
OK | Хүснэгтийн асуулт 8: ['bataa', 'bold', 'saraa', 'tsetseg']


| № | Асуулт | Query | Программын хариу | Гараар шалгасан |
|---|---|---|---|---|
| 1 | Болдын хүүхдүүд | `parent('bold',X)` | anar, zolboo (2) | Тохирсон |
| 2 | Temuulen-ийн өвөө эмээ | `grandparent(X,'temuulen')` | bold, tsetseg (2) | Тохирсон |
| 3 | Munkh-ийн авга/нагац ах нар | `uncle(X,'munkh')` | bold, tomor (2) | Тохирсон |
| 4 | Enkhee-ийн үеэлүүд | `cousin(X,'enkhee')` | anar, munkh, naraa, zolboo (4) | Тохирсон |
| 5 | Үеэлтэй хүмүүс | `has_cousin(X)` | anar, enkhee, munkh, naraa, zolboo (5) | Тохирсон |
| 6 | Хүүхэдгүй хүмүүс | `childless(X)` | enkhee, munkh, naraa, temuulen, zolboo (5) | Тохирсон |
| 7 | Temuulen-ийн элэнц өвөө эмээ | `great_grandparent(X,'temuulen')` | bataa, saraa (2) | Тохирсон |
| 8 | Ач, зээтэй хүмүүс | `has_grandchild(X)` | bataa, bold, saraa, tsetseg (4) | Тохирсон |

**Үгүйсгэл яагаад энэ дарааллаар бичигдсэн бэ?** `person(X)` нь X-ийн боломжит
утгыг мэдэгдэж буй 15 хүнээр хязгаарлаж, дараах `~has_child(X)` тухайн хүнийг шалгана.
Эерэг нөхцөлөөр холбогдоогүй хувьсагчтай үгүйсгэл нь unsafe: Datalog-ийн аюулгүй
дүрмийн шаардлагыг хангахгүй эсвэл хөдөлгүүрээс шалтгаалж алдаа өгч болно.
Prolog-д `\+ has_child(X)`-ийг X холбоогүй байхад шалгавал "ямар нэг хүүхэдтэй хүн
байна уу?" гэж оролдож, нэг нь олдсон даруй бүхэлдээ бүтэлгүйтэнэ;
хүүхэдгүй хүмүүсийг жагсаахгүй.

**Яагаад `~parent(X,Y)` биш вэ?** Хүүхэдгүй гэдэг нь `¬∃y parent(x,y)`.
`has_child(X)` нь эхлээд тэр орших нөхцөлийг нэг аргументтай болгож, дараа нь
үгүйсгэнэ. `¬parent(x,y)` нь нэг тодорхой y-ийн эцэг эх биш гэдгийг л хэлнэ;
бусад хүүхэд байж болно. Y-г эерэг нөхцөлөөр холбохгүйгээр шууд үгүйсгэх нь
аюулгүй биш.

**CWA-ийн хязгаар:** `childless(munkh)` үнэн гарах нь санд Munkh-ийн хүүхэд
бүртгэгдээгүй гэсэн утгатай. Бодит амьдралд хүүхэдгүйг баталсан гэсэн үг биш.

## Алхам 8. Prolog хувилбар ба харьцуулалт

Ижил хавтас дахь **`family.pl`** нь 2–7-р алхам болон хуримын ургийн нэмэлт
дүрмүүдийг агуулна. `family_tests.pl` нь SWI-Prolog-ийн автомат шалгалт.
Шинэ гишүүн нэмэхийг `add_dulguun/0`, буцаахыг `remove_dulguun/0` гүйцэтгэнэ.

SWI-Prolog суусан үед:

```text
swipl -s family.pl
?- findall(P-C, parent(P,C), L), length(L,N).
N = 18.
?- ancestor(bataa,temuulen).
true.
?- childless(temuulen).
true.
?- childless(bataa).
false.
?- findall(Y, sibling(oyuna,Y), L), length(L,N), sort(L,S).
L = [bold,tomor,bold,tomor], N = 4, S = [bold,tomor].
?- setof(X, ancestor(X,temuulen), S).
S = [anar,bat,bataa,bold,saraa,tsetseg].
?- add_dulguun, setof(X,sibling(X,temuulen),S), remove_dulguun.
S = [dulguun].
```

Бүх Prolog шалгалтыг командын мөрөөс ажиллуулах:

```text
swipl -q -s family_tests.pl -g run_tests -t halt
```

SWI-Prolog суулгаагүй бол [SWISH](https://swish.swi-prolog.org/)-д `family.pl`-ийн
агуулгыг оруулаад query хэсэгт асуултаа бичиж болно. Дээрх код блок нь жишээ
асуулт, хүлээгдэх хариуг тайлбарласан; бодитоор ажиллуулсан Prolog үр дүнг
доорх validation cell харуулна.

| Ойлголт | pyDatalog | Prolog |
|---|---|---|
| Баримт | `+parent('bataa','bold')` | `parent(bataa,bold).` |
| Дүрэм | `father(X,Y) <= parent(X,Y) & male(X)` | `father(X,Y) :- parent(X,Y), male(X).` |
| Хоёр нөхцөл | `&` | `,` |
| Үгүйсгэл | `~has_child(X)` | `\+ has_child(X)` |
| Тэнцүү биш | `(X != Y)` | `X \= Y` |
| Давхардал | Set semantics: нэг tuple нэг удаа | Баталгааны зам бүрээр хариу гарна |
| Рекурс | Энэ Datalog сангийн төгсгөлтэй домэйнд тогтмол цэг | Ердийн хайлт нь depth-first; clause/goal дараалал чухал |

`sibling(oyuna,Y)` нь `bataa`, `saraa` гэсэн хоёр эцэг эхээр тус бүр батлагдана.
Тиймээс Prolog-ийн `findall` нь `bold`, `tomor` хоёрыг тус бүр хоёр удаа,
нийт **4** удаа авна. `sort/2` эсвэл `setof/3` хэрэглэвэл **2** ялгаатай хүн болно.
pyDatalog нь дүгнэсэн `(oyuna,bold)` зэрэг tuple-ийг нэг удаа хадгалдаг тул
хоёр баталгаа нь хоёр өөр хариу болохгүй.

In [8]:
# Энэ үр дүнг тусдаа SWI-Prolog хөдөлгүүрээр ажиллуулсан шалгагч үүсгэнэ.
import json
from pathlib import Path
validation_path = Path('output/lab4_fol/prolog_validation.json')
if validation_path.exists():
    validation = json.loads(validation_path.read_text(encoding='utf-8'))
    print('Prolog хөдөлгүүр:', validation['engine'])
    print('Prolog тест:', validation['tests'])
    print('Хоёр хэлний бүрэн харилцааны харьцуулалт:', validation['relations_compared'])
    print('Үр дүн:', validation['status'])
    print('Prolog жишээ асуултын бодит хариу:')
    for label, value in validation['examples'].items():
        print(' ', label, '=', value)
else:
    print('Хадгалсан Prolog validation файл алга. family_tests.pl-ийг дээрх командаар ажиллуулна.')

Prolog хөдөлгүүр: SWI-Prolog swi(10,1,18,[]) (official swipl-wasm 8.2.3)
Prolog тест: 45 PL-Unit tests passed
Хоёр хэлний бүрэн харилцааны харьцуулалт: 25
Үр дүн: PASS: Prolog, pyDatalog, independent Python reference agree
Prolog жишээ асуултын бодит хариу:
  parent count = 18
  sibling proofs = {'proofs': ['bold', 'tomor', 'bold', 'tomor'], 'count': 4, 'unique': ['bold', 'tomor']}
  ancestor(bataa,temuulen) = True
  childless(temuulen) = True
  childless(bataa) = False


## Нэмэлт 1. Нөхөр эхнэр ба хуримын ураг

In [9]:
# Нэг хүүхдийн өөр эцэг эхүүд: ∀x∀y∀z ((parent(x,z) ∧ parent(y,z) ∧ x≠y) → spouse(x,y)).
spouse(X,Y) <= parent(X,Z) & parent(Y,Z) & (X != Y)
# Авга/нагац эгчийн эрэгтэй хань: ∀x∀y∀z ((aunt(z,y) ∧ spouse(x,z) ∧ male(x)) → uncle_in_law(x,y)).
uncle_in_law(X,Y) <= aunt(Z,Y) & spouse(X,Z) & male(X)
# Авга/нагац ахын эмэгтэй хань: ∀x∀y∀z ((uncle(z,y) ∧ spouse(x,z) ∧ female(x)) → aunt_in_law(x,y)).
aunt_in_law(X,Y) <= uncle(Z,Y) & spouse(X,Z) & female(X)

check('Хосын чиглэлтэй холбоо', count('spouse(X,Y)'), 10)
check('Таван хос', sorted((x,y) for x,y in answers('spouse(X,Y)') if x < y),
      [('anar','bat'), ('bataa','saraa'), ('bold','tsetseg'), ('dorj','oyuna'), ('ganaa','tomor')])
check('Авга эгчийн нөхөр', answers('uncle_in_law(X,Y)'),
      [('dorj','anar'), ('dorj','enkhee'), ('dorj','zolboo')])
check('Авга эгчийн нөхрийн холбооны тоо', count('uncle_in_law(X,Y)'), 3)
check('Авга ахын эхнэр', answers('aunt_in_law(X,Y)'),
      [('ganaa','anar'), ('ganaa','munkh'), ('ganaa','naraa'), ('ganaa','zolboo'),
       ('tsetseg','enkhee'), ('tsetseg','munkh'), ('tsetseg','naraa')])
check('Авга ахын эхнэрийн холбооны тоо', count('aunt_in_law(X,Y)'), 7)

# Хоёр хөдөлгүүрийн харьцуулалтад ашиглах үндсэн сангийн snapshot.
BASE_QUERIES = {p: f'{p}(X)' for p in
                ('male','female','person','has_child','childless','has_cousin','has_grandchild')}
BASE_QUERIES.update({p: f'{p}(X,Y)' for p in
                    ('parent','father','mother','son','daughter','grandparent','sibling',
                     'brother','sister','uncle','aunt','cousin','ancestor','descendant',
                     'great_grandparent','spouse','uncle_in_law','aunt_in_law')})
BASE_RESULTS = {p: answers(q) for p,q in BASE_QUERIES.items()}

OK | Хосын чиглэлтэй холбоо: 10
OK | Таван хос: [('anar', 'bat'), ('bataa', 'saraa'), ('bold', 'tsetseg'), ('dorj', 'oyuna'), ('ganaa', 'tomor')]
OK | Авга эгчийн нөхөр: [('dorj', 'anar'), ('dorj', 'enkhee'), ('dorj', 'zolboo')]
OK | Авга эгчийн нөхрийн холбооны тоо: 3
OK | Авга ахын эхнэр: [('ganaa', 'anar'), ('ganaa', 'munkh'), ('ganaa', 'naraa'), ('ganaa', 'zolboo'), ('tsetseg', 'enkhee'), ('tsetseg', 'munkh'), ('tsetseg', 'naraa')]
OK | Авга ахын эхнэрийн холбооны тоо: 7


`spouse`-д хамтын хүүхэд гэсэн нотолгоо хэрэгтэй тул хүүхэдгүй хосыг илрүүлж
чадахгүй. Мөн хамт хүүхэдтэй хүмүүс бодит амьдралд заавал гэрлэсэн байдаггүй;
энэ нь зөвхөн даалгаврын загварын таамаглал юм. Үүнийг өргөжүүлэхдээ
`married(a,b)` эсвэл `partner(a,b)` баримт нэмээд хоёр чиглэлтэй spouse дүрэм
бичнэ: `spouse(X,Y) :- married(X,Y).` ба `spouse(X,Y) :- married(Y,X).`
Ингэснээр хүүхэдгүй хосын холбоог ч мэдлэгийн санд тусгаж болно.

## Нэмэлт 2. Шинэ гишүүн нэмэх, FOL томъёо

In [10]:
# Зөвхөн гурван баримт нэмнэ; дүрэм өөрчлөхгүй.
+male('dulguun')
+parent('bat','dulguun')
+parent('anar','dulguun')
try:
    check('Шинэ хүний тоо', count('person(X)'), 16)
    check('Шинэ эцэг эхийн холбоо', count('parent(X,Y)'), 20)
    check('Temuulen-ийн шинэ ах дүү', names("sibling(X,'temuulen')"), ['dulguun'])
    check('Dulguun-ийн өвөг дээдэс', names("ancestor(X,'dulguun')"),
          ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg'])
    check('Хоёр хүүхдийн өвөг ижил', names("ancestor(X,'dulguun')"), names("ancestor(X,'temuulen')"))
    check('Шинэ ах дүүгийн хосын тоо', count('sibling(X,Y)'), 12)
    check('Шинэ өвгийн холбоо', count('ancestor(X,Y)'), 38)
    check('Шинэ хүүхэдгүй хүмүүсийн тоо', count('childless(X)'), 6)
finally:
    # Дахин ажиллуулахад үндсэн 15 хүний шалгалт хэвээр байлгана.
    -parent('bat','dulguun')
    -parent('anar','dulguun')
    -male('dulguun')

check('Үндсэн санг бүрэн сэргээсэн', {p: answers(q) for p,q in BASE_QUERIES.items()} == BASE_RESULTS, True)
print('Үндсэн сан: 15 хүн, 18 parent холбоо.')

OK | Шинэ хүний тоо: 16
OK | Шинэ эцэг эхийн холбоо: 20
OK | Temuulen-ийн шинэ ах дүү: ['dulguun']
OK | Dulguun-ийн өвөг дээдэс: ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
OK | Хоёр хүүхдийн өвөг ижил: ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
OK | Шинэ ах дүүгийн хосын тоо: 12
OK | Шинэ өвгийн холбоо: 38
OK | Шинэ хүүхэдгүй хүмүүсийн тоо: 6
OK | Үндсэн санг бүрэн сэргээсэн: True
Үндсэн сан: 15 хүн, 18 parent холбоо.


**father**

\[\forall x\forall y((parent(x,y)\land male(x))\to father(x,y)).\]

**grandparent**

\[\forall x\forall y\forall z((parent(x,z)\land parent(z,y))\to grandparent(x,y)).\]

Эквивалент:
\[\forall x\forall y((\exists z(parent(x,z)\land parent(z,y)))\to grandparent(x,y)).\]

**ancestor**-ийн суурь ба рекурсив хоёр дүрмийн хамтатгасан томъёо Алхам 6-д бий.
Horn дүрмийн бүх чөлөөт хувьсагчийг ∀-аар хаана. Толгойд гараагүй биеийн
хувьсагчийг antecedent дотор ∃-аар илэрхийлж болно; энэ нь дээрх эквивалент
хувиргалт бөгөөд толгойд ∃ шинэ хүн үүсгэж байгаа хэрэг биш.

**“Хүн бүр эцэг эхтэй”**

\[\forall x(person(x)\to\exists p(person(p)\land parent(p,x))).\]

Энэ нь ядаж нэг эцэг эх байна гэсэн томъёо. Хоёр өөр эцэг эх гэдгийг тусгайлан
шаардвал хоёр ∃ хувьсагч болон тэнцүү биш нөхцөл нэмэх шаардлагатай.
Өгөгдсөн сангийн CWA/хамгийн бага загварт `person(bataa)` үнэн боловч
`parent(p,bataa)` гэсэн гэрч огт байхгүй тул бүх нийтийн өгүүлбэр худал.
Нээлттэй ертөнцийн FOL ойлголтоор мэдээлэл байхгүй нь бодит эцэг эхгүйг
батлахгүй; энэ сангаас тэр өгүүлбэрийг баталж болохгүй гэсэн үг.

In [11]:
check('Bataa бол хүн', count("person('bataa')"), 1)
check('Bataa-ийн бүртгэлтэй эцэг эх', names("parent(X,'bataa')"), [])
no_recorded_parents = sorted(set(names('person(X)')) - {c for p,c in answers('parent(X,Y)')})
check('Эцэг эх нь бүртгэгдээгүй хүмүүс', no_recorded_parents,
      ['bat', 'bataa', 'dorj', 'ganaa', 'saraa', 'tsetseg'])

OK | Bataa бол хүн: 1
OK | Bataa-ийн бүртгэлтэй эцэг эх: []
OK | Эцэг эх нь бүртгэгдээгүй хүмүүс: ['bat', 'bataa', 'dorj', 'ganaa', 'saraa', 'tsetseg']


## Тайлангийн төгсгөл: асуултын хүснэгт ба үгээр хариулах хэсэг

| № | Асуулт | Гараар гаргасан ба программтай тохирсон хариу | Гараар шалгах үндэслэл |
|---|---|---|---|
| 1 | Болдын хүүхдүүд | anar, zolboo | Bold–Tsetseg-ийн хоёр хүүхэд |
| 2 | Temuulen-ийн өвөө эмээ | bold, tsetseg | Anar-ийн эцэг эх; Bat-ийн эцэг эх өгөгдөөгүй |
| 3 | Munkh-ийн авга/нагац ах нар | bold, tomor | Munkh-ийн эх Oyuna-ийн эрэгтэй ах дүү |
| 4 | Enkhee-ийн үеэлүүд | anar, munkh, naraa, zolboo | Tomor-ийн ах дүү Bold, Oyuna нарын хүүхдүүд |
| 5 | Үеэлтэй хүмүүс | 5: anar, enkhee, munkh, naraa, zolboo | Үе 3-ын гурван өрхийн таван хүүхэд |
| 6 | Хүүхэдгүй хүмүүс | enkhee, munkh, naraa, temuulen, zolboo | Эдгээр 5 нэр parent-ийн эхний аргументад гарахгүй |
| 7 | Temuulen-ийн элэнц өвөө эмээ | bataa, saraa | Temuulen ← Anar ← Bold ← Bataa/Saraa |
| 8 | Ач, зээтэй хүмүүс | bataa, bold, saraa, tsetseg | Хоёр parent холбоо дараалсан замтай 4 хүн |

**Алхам 4: 9 нэмэлт хос яагаад үүссэн бэ?** `X != Y`-г хасахад эцэг эх нь
бүртгэгдсэн `anar, bold, enkhee, munkh, naraa, oyuna, temuulen, tomor, zolboo`
гэсэн 9 хүн тус бүр өөртэйгөө ах дүү болно. Иймээс 10 + 9 = 19.

**Алхам 4: Z-ийн квантор.** Horn дүрмийг хаахад ∀z; харин antecedent дотор
∃z-аар бичсэн эквивалент хэлбэрт Z нь завсрын эцэг эхийн гэрч. Энэ хоёр
тайлбар зөрчилдөхгүй.

**Алхам 8: яагаад Prolog-д 4, pyDatalog-д 2 вэ?** Oyuna-ийн ах дүү Bold,
Tomor хоёр тус бүр Bataa болон Saraa-аар батлагдана. Prolog `findall` бүх
баталгааны замыг авна; pyDatalog ялгаатай tuple-үүдийг авна. `sort` эсвэл
`setof` ашиглавал Prolog-ийн ялгаатай хариу мөн 2 болно.

**Хэлэлцүүлэг 1. CSP ба мэдлэгийн сан.** CSP нь хувьсагч бүрт домэйны утга оноож,
бүх хязгаарлалтыг хангах шийдлийг хайна. Энэ сан нь баримт, дүрмээс холбоо
дүгнэж, нэг сангаас эцэг, үеэл, өвөг зэрэг олон query-д хариулна. Асуулт
солигдоход баримтуудыг дахин загварчлах шаардлагагүй.

**Хэлэлцүүлэг 2. Баримт эсвэл дүрэм.** Одоогийн 12 `grandparent` холбоог
гараар бичвэл 12 баримт шаардлагатай. Нэг ерөнхий дүрэм бүх 12-ыг гаргана.
Шинэ хүн нэмэхэд үндсэн parent баримтыг л шинэчилж, дүрмийг дахин ашиглах нь
илүү хэмнэлттэй; дүгнэсэн холбоог гараар шинэчлэх шаардлагагүй.

**Хэлэлцүүлэг 3. Дутуу мэдээлэл.** Munkh-ийн хүүхэд санд ороогүй байвал
`childless('munkh')` үнэн гарна. Энэ нь сангийн CWA-ийн хариу бөгөөд бодит
амьдралын баталгаа биш.

**Хэлэлцүүлэг 4. Хоёр хөдөлгүүр ба дүрмийн хамрах хүрээ.** Prolog-ийн ердийн
depth-first хайлтад зүүн рекурс асуудалтай, `sibling`-ийн хоёр баталгаа
давхардсан хариу өгнө. pyDatalog энэ төгсгөлтэй, функцгүй дүрмийн санд
давхардалгүй харилцааг дүгнэнэ. Үндсэн uncle/aunt нь цусан төрөл л хамарна;
хуримын ургийг нэмэлт дүрмээр хамруулсан. Хүүхэдгүй хос, үрчлэлт, хойд эцэг эх,
насны ялгаа зэрэг мэдээлэл тусад нь загварчлагдаагүй.

**Дүгнэлт.** 15 хүний 18 эцэг эхийн баримтаас 12 өвөө эмээ, 10 ах дүү,
16 үеэл, 32 өвгийн чиглэлтэй холбоо гарлаа. Үгүйсгэл нь мэдэгдэж буй домэйнд,
CWA нөхцөлтэй хэрэглэгдэв. Dulguun-ийг гурван баримтаар нэмэхэд дүрэм
өөрчлөхгүйгээр шинэ холбоо дүгнэгдэж, туршилтын дараа үндсэн сан сэргэсэн.

**Эх сурвалж:** хэрэглэгчийн өгсөн `Lab6_FOL_Заавар.pdf`, 1–10-р хуудас.
SWI-Prolog шалгалтад [албан ёсны WebAssembly багц](https://github.com/SWI-Prolog/npm-swipl-wasm)
ашигласан; хэрэглэгч ердийн SWI-Prolog эсвэл SWISH-ээр `family.pl`-ийг ажиллуулж болно.

In [12]:
display(Markdown('| Харилцаа | Тоо |\n|---|---|\n' + '\n'.join(
    f'| `{predicate}` | {len(rows)} |' for predicate,rows in BASE_RESULTS.items())))
print(f'Notebook-ийн {len(checks)} шалгалт амжилттай. Үндсэн сан сэргэсэн.')

| Харилцаа | Тоо |
|---|---|
| `male` | 8 |
| `female` | 7 |
| `person` | 15 |
| `has_child` | 10 |
| `childless` | 5 |
| `has_cousin` | 5 |
| `has_grandchild` | 4 |
| `parent` | 18 |
| `father` | 9 |
| `mother` | 9 |
| `son` | 10 |
| `daughter` | 8 |
| `grandparent` | 12 |
| `sibling` | 10 |
| `brother` | 6 |
| `sister` | 4 |
| `uncle` | 8 |
| `aunt` | 3 |
| `cousin` | 16 |
| `ancestor` | 32 |
| `descendant` | 32 |
| `great_grandparent` | 2 |
| `spouse` | 10 |
| `uncle_in_law` | 3 |
| `aunt_in_law` | 7 |

Notebook-ийн 70 шалгалт амжилттай. Үндсэн сан сэргэсэн.
